# 1. Data Collection with the SpaceX REST API
**Goal:** collect Falcon 9 launch records from the SpaceX API and build one clean dataframe.

Flow: request `/launches/past` -> keep single-core, single-payload Falcon 9 launches -> look up booster, site, payload and core details by ID -> assemble dataframe -> save CSV.

In [ ]:
import requests, pandas as pd, numpy as np, datetime
pd.set_option('display.max_columns', None)

## Request the launch data
The live endpoint is `https://api.spacexdata.com/v4/launches/past`. The course also provides a frozen copy of the response, which keeps results reproducible.

In [ ]:
# Live API call (key phrase of this lab)
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
print('live API status:', response.status_code)

# Static snapshot used for reproducible results
static_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json"
data = pd.json_normalize(requests.get(static_url).json())
data.head()

## Keep only the columns we need and filter launches
Remove launches with more than one core or payload, and restrict to dates up to 2020-11-13.

In [ ]:
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])
data['date'] = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= datetime.date(2020, 11, 13)]
print(len(data), 'launches kept')

## Look up details by ID
Each ID is queried once and cached, so the number of API calls stays small.

In [ ]:
def lookup(endpoint, ids):
    cache = {}
    for i in set(ids):
        if i is not None:
            cache[i] = requests.get(f"https://api.spacexdata.com/v4/{endpoint}/{i}").json()
    return cache

rockets   = lookup('rockets', data['rocket'])
pads      = lookup('launchpads', data['launchpad'])
payloads  = lookup('payloads', data['payloads'])
core_ids  = [c['core'] for c in data['cores']]
cores     = lookup('cores', core_ids)

In [ ]:
rows = []
for _, r in data.iterrows():
    core, pad, pl = r['cores'], pads[r['launchpad']], payloads[r['payloads']]
    cinfo = cores.get(core['core'], {})
    rows.append({
        'FlightNumber': r['flight_number'],
        'Date': r['date'],
        'BoosterVersion': rockets[r['rocket']]['name'],
        'PayloadMass': pl['mass_kg'],
        'Orbit': pl['orbit'],
        'LaunchSite': pad['name'],
        'Outcome': str(core['landing_success']) + ' ' + str(core['landing_type']),
        'Flights': core['flight'],
        'GridFins': core['gridfins'],
        'Reused': core['reused'],
        'Legs': core['legs'],
        'LandingPad': core['landpad'],
        'Block': core['block'],
        'ReusedCount': cinfo.get('reuse_count'),
        'Serial': cinfo.get('serial'),
        'Longitude': pad['longitude'],
        'Latitude': pad['latitude'],
    })
launch_df = pd.DataFrame(rows)
launch_df.head()

## Keep Falcon 9 only and handle missing payload mass
Falcon 1 launches are removed. Missing `PayloadMass` values are replaced by the column mean.

In [ ]:
data_falcon9 = launch_df[launch_df['BoosterVersion'] == 'Falcon 9'].copy()
data_falcon9['FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
print('missing before:\n', data_falcon9.isnull().sum())
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].replace(np.nan, data_falcon9['PayloadMass'].mean())
print('missing PayloadMass after:', data_falcon9['PayloadMass'].isnull().sum())
data_falcon9.to_csv('dataset_part_1.csv', index=False)
data_falcon9.head()